## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Vivian Le

**Date:** 9/24/26

Your Phase 1 solution to the problem goes here.

Q1.

#1: Both are types of supervised learning; the distinction is based on the nature of the dependent variable. In regression analysis, the output value will be numeric and continuous (for example, price, temperature), and the algorithm will try to predict a numeric value, and its error will be determined by the distance from the actual value (SSE). In classification, the dependent variable is categorical (for example, mine type 1-5), and the algorithm predicts the class label or class probability, and the measure of its performance is how frequently it makes incorrect classifications and in which way (accuracy, confusion table).

#2: A confusion table is a tabular representation where row index is the true label of the class and column index is the predicted class label with cell values being the number of observations of true classes that have been classified into each class. The cells on the diagonal represent correctly classified samples in comparison to the rest of the cells contain errors. The importance of the confusion matrix is that it informs us about the types of errors made by the classification method rather than simply the number of errors made. This distinction is particularly useful in cases where false positives and false negatives cost differently.


#3: Sum of squares of error (SSE) is a measure that represents the total squared deviation of actual results from the predicted results. In other words, this measure represents how well the model predicts results; the lower the sum, the better. Square of an error always gives the positive result and greatly punishes for big errors in contrast to small ones. As a sum, it increases with the sample size, hence it should be used only to compare models based on the same sample.

#4: Overfitting happens when the model is flexible to an extent where it picks up the noise in the data, instead of the pattern; it does exceptionally well with the training data and not so good with the new data. When k is extremely low (such as k=1), there is overfitting with kNN, as the predictions rely on just one, maybe noisier, neighbor. Underfitting refers to the condition where the model is too simple and does not grasp the actual pattern in the data; the model does not perform well with the training data nor with the new data. In kNN, underfitting happens when k is extremely high.

#5: If we had evaluated our model using the same data that we used for training, we would have found the most flexible model to be the best one since in the case of k=1, the model perfectly fits the training data and achieves 100% training accuracy and an SSE of 0 despite being overfitted. The test set which consists of data that the model has not been exposed to during the training phase acts as a substitute for evaluating how well the model performs on unseen data. Evaluating test accuracy and SSE of our model as a function of k allows us to determine that test error is large for small k (overfitting), large for large k (underfitting), and minimal somewhere in between. The k corresponding to minimal test error best represents the best generalizing model instead of the one which just memorized the training data.

#6: Starting with the class label, the main strength is simplicity. It gives one direct, actionable answer that is easy to interpret and easy to evaluate with accuracy or a confusion table. The main weakness is that it hides uncertainty. A close 51 to 49 vote and a unanimous 100 to 0 vote produce the same label, so you can't tell a confident prediction from a near coin flip. It is also strict when it comes to the choice of a decision rule that might not be appropriate if different types of errors have different costs. On the other hand, the probability distribution's major advantage is that it shows the confidence of the model. This enables the user to choose a cut-off point based on the cost of each type of error, such as flagging a specific location if the probability of a mine there exceeds 20 percent. Another advantage is the possibility to use techniques such as ROC-curves and ranking of predictions according to the probability. The disadvantages are that it is less clear and more difficult to interpret, it still requires a cut-off point or a decision rule to transform into a decision and the probabilities might not reflect the real frequency of an event. In particular, when using kNN, the probability estimates are very rough. They are limited to intervals of size one divided by k and come from a few nearest neighbors only.



In [5]:
# Q2:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 1.
df = pd.read_csv('./data/land_mines.csv')
print(df.groupby('mine_type').mean())

print(df['soil'].value_counts().sort_index())
print(df['mine_type'].value_counts().sort_index())
print(df.describe())
df.info()

# 2.
X = df[['voltage', 'height', 'soil']]
y = df ['mine_type']
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.5, random_state=42)

# 3.
accuracies = []
for k in range (1,41):
    model = KNeighborsClassifier (n_neighbors=k).fit(X_train, y_train)
    accuracies.append(model.score(X_test, y_test))

best_k = np.argmax(accuracies) + 1
print('Optimal k:', best_k, '| Test Accuracy:', max(accuracies))

#4.
model = KNeighborsClassifier(n_neighbors=best_k).fit(X_train, y_train)
y_hat = model.predict(X_test)
pd.crosstab(y_test, y_hat, rownames=['Actual'], colnames=['Predicted'])

            voltage    height      soil
mine_type                              
1          0.296463  0.495519  0.492958
2          0.721123  0.487013  0.508571
3          0.402408  0.527548  0.496970
4          0.345365  0.506887  0.503030
5          0.379598  0.530070  0.516923
soil
0.0    59
0.2    51
0.4    56
0.6    57
0.8    58
1.0    57
Name: count, dtype: int64
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data c

Predicted,1,2,3,4,5
Actual,,,,,
1,17,0,18,2,0
2,0,25,4,4,0
3,8,3,16,2,3
4,12,5,10,7,2
5,8,2,12,6,3


Part 1: The dataset contains 338 observations with no missing values. There are almost equal number of instances of each of the five mines (65-71 instances). Features have been scaled between 0 and 1. Soil can take just six values and hence it is a categorical feature. The most significant feature is voltage: mine of type 2 has a much higher average value of voltage (about 0.72) compared to others (about 0.30-0.40). Height and soil are almost identical in all five types of mines.

Part 3: For all values of k from 1 to 40, kNN classifier was fitted and accuracy was calculated using the test set. The value of k that gave the highest test accuracy was chosen, which was k = 2 with an accuracy of about 40%. Lower values of k will lead to overfitting because of high dependence on one or two noisy instances. Higher values of k will result in underfitting due to averaging over a lot of points.

Part 4: Accuracy of the test in general is about 40%, which is higher than the probability of making guesses randomly (20%). Type 2 is the most accurate one (25 out of 33 or 76%), since its voltage is much higher compared to the rest. Type 3 is accurate 50% of the time and type 1 46%. Types 4 and 5 are poor ones at 19% and 10%. Types 1, 3, 4, and 5 are very similar, and the model gets them mixed up, with type 3 acting as a catch-all, and 60 points were predicted as type 3, but only 16 actually were.

#5. I would first warn that at certain times, the model is inaccurate. In the dataset, Type 1 means no mine and types 2 through 5 are different kinds of mines. The most dangerous mistake is calling a real mine "no mine," and the model did that for 28 of the 132 real mines in the test set. A type can also have many correct predictions and still be unreliable. For example, type 3 had 16 correct predictions, but 60 test points were predicted as type 3, so most of the predictions were wrong. After telling the individual this information, I would advise using this model only as a screening and planning tool, never as the final decision. A "no mine" prediction should never be a reason to skip clearance, and every location should still be checked with full safety procedures. Type 2 mines are identified correctly approximately 76% of the time, so a team could plan for that mine type with some confidence. Types 3, 4, and 5 are constantly confused with each other, so any of those predictions should be treated as "some type of mine" and handled with the most cautious procedure. The model is best for deciding which areas to clear first or what equipment to bring. Overall, the model labeled 28 of 132 real mines as "no mine," so it should never replace full safety procedures. You should use it to prioritize and plan, trusting type 2 predictions but treating types 3 to 5 as simply "a mine."

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** fc32841
- **AI tool and model used:** ChatGPT -5.6 Sol

### *Prompts used

1. Give me feedback on my Q1 without rewriting my answers for me.
2. Here's my Q2 code, run it and check for missing values, correctness, and  anything that could be improved but do not rewrite it

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. #1 — Regression vs. classification: Strong. You clearly identify the main distinction: regression predicts a quantitative/numeric outcome, while classification predicts a categorical outcome. Your examples are appropriate, and mentioning class probabilities is especially useful because it connects directly to #6. The main technical caveat is that saying regression error “will be determined” by SSE makes SSE sound like the only possible regression metric. It is one metric, and it is clearly relevant to your assignment, but regression can be evaluated other ways. Also, “continuous” is slightly narrower than “quantitative/numeric,” although it is probably what your course expects.
#2 — Confusion table: Very strong. You explain both what the table contains and, importantly, why it is useful. Your discussion of diagonal versus off-diagonal cells shows that you understand how to interpret one rather than just define it. The false-positive/false-negative point directly answers what a confusion table helps us understand about performance. One edge case: actual labels are not universally rows and predicted labels columns; some conventions reverse them. If that orientation matches your course, there is no problem. Otherwise, the definition could be considered unnecessarily absolute. The phrase "in comparison to the rest of the cells contain errors" is grammatically incomplete; replacing "in comparison to" with ", while" makes it a clear contrast between correct predictions on the diagonal and errors in the other cells.
#3 — SSE: Strong. You correctly describe SSE as the sum of squared differences between observed and predicted values, explain that lower SSE indicates closer predictions when making an appropriate comparison, and recognize that squaring prevents positive and negative errors from canceling while penalizing large errors more heavily. Your sample-size observation is also good. The only nuance is that “the lower the sum, the better” assumes you're comparing models evaluated on the same data (or otherwise making a fair comparison), which you essentially acknowledge in your final sentence.
#4 — Overfitting and underfitting: Strong. Your definitions are accurate, and connecting them specifically to kNN makes sense because the next question asks about choosing \(k\). You correctly associate very small \(k\) with greater model flexibility/overfitting and very large \(k\) with excessive smoothing/underfitting. The only edge case is that these are tendencies rather than absolute rules for every conceivable dataset. Your conceptual explanation is right.
#5 — Train/test split and choosing \(k\): Very strong, with one point to check. Now that you've shown the exact prompt, your discussion of selecting \(k\) based on the test set is completely aligned with the assignment. You do not need to introduce validation sets or cross-validation unless your instructor has covered them, because the question explicitly asks why choosing \(k\) using test accuracy/SSE helps. Your explanation of why training performance favors overly flexible models is particularly good.
The only thing I'd verify is your sentence that \(k=1\) gives “100% training accuracy and an SSE of 0.” Those statements correspond naturally to classification and regression respectively under standard kNN training evaluation where an observation can be its own nearest neighbor. Since the question itself says “accuracy or SSE,” your professor may deliberately want you to address both. There are edge cases involving duplicate observations, conflicting labels, ties, or implementations that exclude the observation itself, so it isn't a completely universal property of every kNN setup. For a standard introductory treatment, though, your reasoning is sound. Also, the familiar high-error-at-both-extremes/U-shaped test-error pattern is a conceptual tendency rather than a mathematical guarantee that every dataset's minimum must occur strictly in the middle.
#6 — Labels vs. probability distributions: Excellent. This is probably your strongest response. The 51–49 versus 100–0 example very clearly demonstrates the information lost when reporting only the winning class. You correctly identify simplicity and actionability as strengths of labels and loss of uncertainty as a weakness. For probabilities, you correctly discuss thresholds, unequal error costs, ranking, ROC curves, and imperfect probability estimates. Your observation that ordinary unweighted kNN probabilities occur in increments of \(1/k\) is also good.
There are two edge cases in #6. Calling predicted probability the model's “confidence” is intuitive and probably fine at this level, but probability and reliable confidence are not necessarily identical when a model is poorly calibrated—which you effectively recognize later when you say probabilities might not reflect real event frequencies. And the \(1/k\) restriction applies to ordinary unweighted kNN; weighted kNN can produce other values.
Overall: I don't see a major conceptual misunderstanding in Q1. #2, #4, #5, and #6 especially demonstrate understanding rather than memorization. The areas I'd actually inspect before submitting are mostly precision: SSE isn't the only regression metric (#1), confusion-matrix orientation can vary (#2), and the \(k=1\) claims in #5 assume the standard kNN setup. None of those require changing the core reasoning of your answers.
2. The only thing I'd consider changing
If the assignment explicitly expects you to check for missing values, I'd consider adding an explicit missing-value check because right now you're relying on df.info().
But technically, you already can see from df.info() that there are none. So if the assignment simply asks you to explore the data, I don't think this is a correctness problem.
There is one conceptual thing worth knowing about soil: its values are 0, .2, .4, .6, .8, 1. Your code treats those values numerically when calculating kNN distance. That may be completely intentional—the dataset appears to have already numerically encoded/scaled the variable. I wouldn't change that without the assignment telling you to.
Bottom line: your Q2 code works, there are 0 missing values, \(k=2\) is correctly selected, test accuracy is ~40.24%, and your confusion table is correctly constructed. I don't see anything you need to fix based on the actual dataset.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Reject | I decided to leave #5, and #6 largely alone due to the feedback received because they were both already accurate and correct. |
| 2 | Accept | I did change #2 as the original sentence had a grammatical error; however, the meaning did not change. |
| 3 | Accept |  In addition, I also decided to make a change to #3 based on the AI's suggestion about technical wording - nonnegative is the right word rather than positive in this situation because 0^2 is 0, which is nonnegative. |
| 4 | Modify |  I did modify the wording of #4 because my initial answer assumed that small k always overfits and large k always underfits. This is incorrect, as the relationship is dependent upon the data. With a small value of k, overfitting is more probable to happen as the predictions are based on such a few number of neighbors. However, with a larger value of k, there is more likelihood of underfitting as the data will have so many neighbors such that all patterns get hidden. |
| 4 | Reject | Suggested that I add a check for missing-value part of the code, but I rejected this part because df.info() already shows that there are 338 non-null entries, meaning that there are not any missing values, so I do not have to include this in my code. Further, the code is correct and accurate. |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

# Your Phase 2 solution to the problem goes here.

Q1.

#1: Both are types of supervised learning; the distinction is based on the nature of the dependent variable. In regression analysis, the output value will be numeric and continuous (for example, price, temperature), and the algorithm will try to predict a numeric value, and its error will be determined by the distance from the actual value (SSE). In classification, the dependent variable is categorical (for example, mine type 1-5), and the algorithm predicts the class label or class probability, and the measure of its performance is how frequently it makes incorrect classifications and in which way (accuracy, confusion table).

#2: A confusion table is a tabular representation where row index is the true label of the class and column index is the predicted class label with cell values being the number of observations of true classes that have been classified into each class. The cells on the diagonal represent correctly classified samples, while the rest of the cells contain errors. The importance of the confusion matrix is that it informs us about the types of errors made by the classification method rather than simply the number of errors made. This distinction is particularly useful in cases where false positives and false negatives cost differently.


#3: Sum of squares of error (SSE) is a measure that represents the total squared deviation of actual results from the predicted results. In other words, this measure represents how well the model predicts results; the lower the sum, the better. Square of an error always gives a nonnegative result and greatly punishes for big errors in contrast to small ones. As a sum, it increases with the sample size, hence it should be used only to compare models based on the same sample.

#4: Overfitting happens when the model is so flexible that it picks up the noise in the data, instead of the pattern; it does exceptionally well with the training data and not so good with the new data. When k is extremely low (such as k=1), there is a greater risk of overfitting with kNN, as the predictions rely on just one neighbor, making them more sensitive to noise or unusual observations. Underfitting refers to the condition where the model is too simple and does not grasp the actual pattern in the data; the model does not perform well with the training data nor with the new data. In kNN,the risk of underfitting increases when k is extremely high.

#5: If we had evaluated our model using the same data that we used for training, we would have found the most flexible model to be the best one since in the case of k=1, the model perfectly fits the training data and achieves 100% training accuracy and an SSE of 0 despite being overfitted. The test set which consists of data that the model has not been exposed to during the training phase acts as a substitute for evaluating how well the model performs on unseen data. Evaluating test accuracy and SSE of our model as a function of k allows us to determine that test error is large for small k (overfitting), large for large k (underfitting), and minimal somewhere in between. The k corresponding to minimal test error best represents the best generalizing model instead of the one which just memorized the training data.

#6: Starting with the class label, the main strength is simplicity. It gives one direct, actionable answer that is easy to interpret and easy to evaluate with accuracy or a confusion table. The main weakness is that it hides uncertainty. A close 51 to 49 vote and a unanimous 100 to 0 vote produce the same label, so you can't tell a confident prediction from a near coin flip. It is also strict when it comes to the choice of a decision rule that might not be appropriate if different types of errors have different costs. On the other hand, the probability distribution's major advantage is that it shows the confidence of the model. This enables the user to choose a cut-off point based on the cost of each type of error, such as flagging a specific location if the probability of a mine there exceeds 20 percent. Another advantage is the possibility to use techniques such as ROC-curves and ranking of predictions according to the probability. The disadvantages are that it is less clear and more difficult to interpret, it still requires a cut-off point or a decision rule to transform into a decision and the probabilities might not reflect the real frequency of an event. In particular, when using kNN, the probability estimates are very rough. They are limited to intervals of size one divided by k and come from a few nearest neighbors only.

In [ ]:
# Q2:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# 1.
df = pd.read_csv('./data/land_mines.csv')
print(df.groupby('mine_type').mean())

print(df['soil'].value_counts().sort_index())
print(df['mine_type'].value_counts().sort_index())
print(df.describe())
df.info()

# 2.
X = df[['voltage', 'height', 'soil']]
y = df ['mine_type']
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.5, random_state=42)

# 3.
accuracies = []
for k in range (1,41):
    model = KNeighborsClassifier (n_neighbors=k).fit(X_train, y_train)
    accuracies.append(model.score(X_test, y_test))

best_k = np.argmax(accuracies) + 1
print('Optimal k:', best_k, '| Test Accuracy:', max(accuracies))

#4.
model = KNeighborsClassifier(n_neighbors=best_k).fit(X_train, y_train)
y_hat = model.predict(X_test)
pd.crosstab(y_test, y_hat, rownames=['Actual'], colnames=['Predicted'])

Part 1: The dataset contains 338 observations with no missing values. There are almost equal number of instances of each of the five mines (65-71 instances). Features have been scaled between 0 and 1. Soil can take just six values and hence it is a categorical feature. The most significant feature is voltage: mine of type 2 has a much higher average value of voltage (about 0.72) compared to others (about 0.30-0.40). Height and soil are almost identical in all five types of mines.

Part 3: For all values of k from 1 to 40, kNN classifier was fitted and accuracy was calculated using the test set. The value of k that gave the highest test accuracy was chosen, which was k = 2 with an accuracy of about 40%. Lower values of k will lead to overfitting because of high dependence on one or two noisy instances. Higher values of k will result in underfitting due to averaging over a lot of points.

Part 4: Accuracy of the test in general is about 40%, which is higher than the probability of making guesses randomly (20%). Type 2 is the most accurate one (25 out of 33 or 76%), since its voltage is much higher compared to the rest. Type 3 is accurate 50% of the time and type 1 46%. Types 4 and 5 are poor ones at 19% and 10%. Types 1, 3, 4, and 5 are very similar, and the model gets them mixed up, with type 3 acting as a catch-all, and 60 points were predicted as type 3, but only 16 actually were.

#5. I would first warn that at certain times, the model is inaccurate. In the dataset, Type 1 means no mine and types 2 through 5 are different kinds of mines. The most dangerous mistake is calling a real mine "no mine," and the model did that for 28 of the 132 real mines in the test set. A type can also have many correct predictions and still be unreliable. For example, type 3 had 16 correct predictions, but 60 test points were predicted as type 3, so most of the predictions were wrong. After telling the individual this information, I would advise using this model only as a screening and planning tool, never as the final decision. A "no mine" prediction should never be a reason to skip clearance, and every location should still be checked with full safety procedures. Type 2 mines are identified correctly approximately 76% of the time, so a team could plan for that mine type with some confidence. Types 3, 4, and 5 are constantly confused with each other, so any of those predictions should be treated as "some type of mine" and handled with the most cautious procedure. The model is best for deciding which areas to clear first or what equipment to bring. Overall, the model labeled 28 of 132 real mines as "no mine," so it should never replace full safety procedures. You should use it to prioritize and plan, trusting type 2 predictions but treating types 3 to 5 as simply "a mine."

### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

The AI feedback helped me make the technical answers more precise. For Q1, I replaced "positive" with "nonnegative" in #3 because the error can be equal to zero. Also, I have made some changes in #4 because in my first version of the answer, there was an idea that small k causes overfitting and large k – underfitting for sure. However, the AI has told me that these are only tendencies and not guarantees. I have kept #2, #5, and #6 almost as they were because I have found the main reasoning correct. One disadvantage was the focus of some of the feedbacks on the edge cases, when there were no errors in my answer, such as the other possible orientation of the confusion table and special case when k=1. For Q2, the AI first made some comments regarding the scaling and stratification of the data, which were of little significance after analyzing the dataset as it was. To verify the solution, I have run the code using the dataset. There were no missing values, errors, and the code has chosen k=2 with test accuracy of about 40.24% and generated a confusion table. The last problem is that the